# 매수 체결대금 30억 원 · 1% 상승 패턴 학습 — Colab A100

`colab_train_xlstm_return_priority.ipynb`를 기반으로 만든 **새 조건 전용** 노트북입니다. 아래 순서대로 실행합니다.

- 진입 후보: **같은 과거 5~60초 구간의 매수 방향 체결대금 ≥ 30억 원 AND 가격 상승률 ≥ 1%**.
- 체결 방향: 거래량이 **음수이면 매도, 그 외에는 매수**. 음수가 없는 데이터도 모두 매수로 처리합니다. 0은 금액에 영향을 주지 않으며, 누적거래량이 변하지 않은 반복 행은 제외합니다.
- 학습 목표: 실제 매수 체결 후 **1~5초 내 현재가가 해당 체결가 이상**이면 +1, 아니면 -1의 별도 PPO 정책 신호를 해당 매수 결정에 연결합니다. 부분체결은 체결 수량으로 가중 평균합니다.
- 결과 미확정: 5초까지 데이터가 없거나 1~5초 구간에 관측이 없으면 패턴 평가에서 제외합니다.
- 관측: **seq_len=2048 시장 행 × 65차원**. 2048초를 의미하지 않습니다.
- 순수익: 수수료·세금·체결 지연을 반영한 NAV 보상과 가치함수 목표를 유지합니다. 패턴 성공률은 순이익 확률과 별개입니다.

미래 가격은 결과 라벨에만 사용합니다. 입력·진입 조건·에피소드 선정은 과거 정보만 사용합니다. 검증과 테스트도 같은 진입 필터를 적용하므로 결과는 해당 조건을 만족하는 구간의 성과입니다.

**준비물:** 수정된 프로젝트 소스, 새 조건으로 재추출한 `extracted_episodes_entry_pattern.tar`, A100 런타임.
노트북만 업로드하면 이전 프로젝트 코드나 기존 NPZ가 자동 변환되지는 않습니다. 3번에서 수정 소스를 준비하고 4번에서 새 TAR 경로를 지정하세요.
처음에는 `MODE='new'`, 새 `RUN_NAME`, 빈 `LOAD_POLICY`로 시작합니다. 이 노트북은 같은 진입 조건과 2048 관측을 가진 체크포인트만 재개·미세조정합니다. 이전 실험 재현에는 원본 노트북을 사용하세요.


## 1. A100 및 시스템 RAM 확인

런타임에서 A100을 선택하세요. [A100은 40GB/80GB 모델이 있으며](https://www.nvidia.com/en-us/data-center/a100/),
[Colab의 GPU 및 시스템 RAM 할당은 세션에 따라 달라집니다](https://research.google.com/colaboratory/faq.html).
GPU 종류와 사용 가능한 RAM을 실제로 확인하여 설정합니다.

In [ ]:
import gc
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time
import torch

GiB = 1024 ** 3

def available_ram_gib():
    entries = dict(line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines())
    return int(entries['MemAvailable'].split()[0]) * 1024 / GiB

if not torch.cuda.is_available() or 'A100' not in torch.cuda.get_device_name(0):
    raise RuntimeError('런타임 → 런타임 유형 변경에서 A100 GPU를 선택하세요.')
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / GiB
if VRAM_GIB < 30:
    raise RuntimeError('이 프로필은 전체 A100용입니다. 작은 MIG 파티션은 별도 설정이 필요합니다.')
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 1)
print(f'{GPU_NAME} | VRAM {VRAM_GIB:.1f} GiB | 여유 RAM {available_ram_gib():.1f} GiB | CPU {CPU_COUNT}')
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')

## 2. Drive 연결 및 학습 의존성 설치

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Colab에 설치된 CUDA용 PyTorch를 유지합니다.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'duckdb>=1.4', 'pandas>=2.0', 'pyarrow>=14', 'python-dotenv>=1.0',
                'gymnasium>=0.29', 'scikit-learn>=1.4', 'tensorboard>=2.14'], check=True)

## 3. 수정된 프로젝트 코드 준비

`entry_pattern.py`와 수정된 추출기·환경·GRPO 학습기가 포함된 소스가 필요합니다. 원격에 수정본이 반영되어 있으면 해당 `REVISION`을 지정하세요. 아직 원격에 반영하지 않았다면 수정한 프로젝트 폴더를 `/content/stock-bot2`에 준비하고 `REVISION`을 비웁니다. 기존 폴더는 자동 덮어쓰지 않습니다.

Private 저장소는 Colab Secret의 `GITHUB_TOKEN`을 사용합니다. 토큰을 URL에 저장하지 않습니다.


In [ ]:
from google.colab import userdata
import tempfile

REPO_PATH = Path('/content/stock-bot2')
REPO_URL = 'https://github.com/gblue1223/stock-bot2.git'
REVISION = ''  # @param {type:"string"}

try:
    github_token = userdata.get('GITHUB_TOKEN')
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    github_token = None

git_env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
try:
    with tempfile.TemporaryDirectory() as temporary:
        if github_token:
            askpass = Path(temporary) / 'askpass.py'
            askpass.write_text('#!/usr/bin/env python3\nimport os,sys\n'
                               'print("x-access-token" if "username" in sys.argv[1].lower() '
                               'else os.environ["COLAB_GIT_TOKEN"])\n')
            askpass.chmod(0o700)
            git_env.update(GIT_ASKPASS=str(askpass), COLAB_GIT_TOKEN=github_token)
        if not REPO_PATH.exists():
            subprocess.run(['git', 'clone', REPO_URL, str(REPO_PATH)], env=git_env, check=True)
        if REVISION.strip() and not (REPO_PATH / '.git').exists():
            raise RuntimeError('직접 업로드한 폴더에는 REVISION을 비우세요. Git checkout에서만 revision을 변경합니다.')
        if REVISION.strip():
            dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_PATH, text=True)
            if dirty.strip():
                raise RuntimeError('수정된 파일이 있어 revision을 바꾸지 않았습니다. 수정본을 먼저 보존하세요.')
            subprocess.run(['git', 'fetch', 'origin', REVISION.strip()], cwd=REPO_PATH, env=git_env, check=True)
            subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO_PATH, check=True)
finally:
    git_env.pop('COLAB_GIT_TOKEN', None)
    github_token = None

required = ['lib/observations.py', 'ai_trader/grpo/environments/execution.py',
            'ai_trader/grpo/evaluation.py', 'ai_trader/grpo/backtest.py',
            'ai_trader/grpo/policy_update_checks.py', 'ai_trader/grpo/runtime_precision.py',
            'ai_trader/grpo/diagnose_likelihood.py', 'ai_trader/grpo/update_diagnostic.py',
            'ai_trader/grpo/diagnose_update_lr.py', 'ai_trader/grpo/entry_pattern.py']
if any(not (REPO_PATH / name).is_file() for name in required):
    raise RuntimeError('이전 프로젝트 코드입니다. 리뷰 수정본이 포함된 REVISION 또는 폴더를 준비하세요.')
os.chdir(REPO_PATH)
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))
from ai_trader.grpo.train_xlstm import TrainingConfig, create_environment, prepare_date_splits
from lib.observations import SCHEMA_VERSION, ObservationBuilder
required_settings = ('execution_observations', 'policy_update_checks', 'rollout_logprob_tolerance',
                     'kl_probe_samples', 'no_trade_max_validations', 'resume_lr', 'capture_update_bundle',
                     'entry_pattern_config')
if SCHEMA_VERSION != 2 or not all(hasattr(TrainingConfig(), key) for key in required_settings):
    raise RuntimeError('노트북과 프로젝트 버전이 맞지 않습니다. 수정본으로 런타임을 다시 준비하세요.')
from ai_trader.grpo.policy_update_checks import check_rollout_likelihood
from ai_trader.grpo.policies.scalping_policy_xlstm import GRPOPolicyE2EXLSTM
if not callable(getattr(GRPOPolicyE2EXLSTM, 'evaluate_actions_with_distribution', None)):
    raise RuntimeError('정책 검사 API가 이전 버전입니다. 코드를 갱신하고 런타임을 재시작하세요.')
if (REPO_PATH / '.git').exists():
    CODE_REVISION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_PATH, text=True).strip()
else:
    CODE_REVISION = 'uploaded-local-snapshot'
source_digest = hashlib.sha256()
for source in sorted([*(REPO_PATH / 'ai_trader/grpo').rglob('*.py'), *(REPO_PATH / 'lib').rglob('*.py')]):
    source_digest.update(source.relative_to(REPO_PATH).as_posix().encode())
    source_digest.update(source.read_bytes())
SOURCE_SHA256 = source_digest.hexdigest()
print('사용 코드:', CODE_REVISION)

## 4. 새 조건으로 재추출한 TAR 준비

기존 NPZ에는 체결 방향 정보가 없어 다시 추출해야 합니다. 아래는 **로컬 프로젝트 폴더에서 실행할 PowerShell 명령**입니다. 이 노트북은 준비된 TAR를 Drive에서 복사하여 해제합니다.

```powershell
python -m ai_trader.grpo.data_extractor --db "C:/Users/user/Workspace/datasets@raw/datasets_all.duckdb" --output_dir data/extracted_episodes_entry_pattern --seq_len 2048 --features 29 --max_steps 300 --price-unit krw --time-start 90000000 --time-end 110000000 --entry-pattern-config config/scalping_v3.example.json --buy-notional-source signed_volume_positive_buy
tar -cf data/extracted_episodes_entry_pattern.tar -C data/extracted_episodes_entry_pattern .
```

`거래량 >= 0`인 실제 체결의 `현재가 × 체결량`만 합산합니다. `누적거래량` 증가량과 체결량의 절댓값이 일치해야 하며, 호가·거래원 갱신에 반복된 거래량을 다시 더하지 않습니다. 첫 원본 행 이전의 체결은 추정하지 않습니다. 전체 누적거래대금이나 매수대기금액으로 대체하지 않습니다.

생성한 TAR를 Drive에 업로드한 뒤 아래 `DRIVE_TAR_PATH`를 지정하세요. TAR의 루트 또는 한 데이터 폴더 안에 `manifest.json`과 그 manifest가 가리키는 NPZ가 있어야 합니다. 필터를 통과한 종목·날짜만 포함되며, 최소 3개 거래일이 필요합니다.

- 로컬 공간은 TAR와 해제본 및 여유 공간을 합산해 검사합니다. 크기는 실제 추출 결과에 따라 달라집니다.
- 같은 VM의 완료된 캐시는 재사용합니다. 불완전한 파일·누락된 NPZ·경로 이탈은 거부합니다.
- 원본 138GB DB를 Colab으로 복사할 필요가 없습니다. 추출은 원본 데이터가 있는 로컬에서 수행합니다.


In [ ]:
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import tarfile


def _archive_name(name):
    path = PurePosixPath(name)
    if not name or path.is_absolute() or '..' in path.parts or '\\' in name or ':' in name:
        raise ValueError(f'Unsafe archive path: {name!r}')
    return path.as_posix()


def _local_target(root, name):
    target = root / _archive_name(name)
    if target.is_symlink() or not target.resolve().is_relative_to(root.resolve()):
        raise ValueError(f'Local cache path escapes its directory: {name}')
    return target


def _read_cache_record(path):
    try:
        record = json.loads(path.read_text(encoding='utf-8'))
        return record if isinstance(record, dict) else {}
    except (FileNotFoundError, ValueError):
        return {}


def _write_cache_record(root, name, record):
    target = _local_target(root, name)
    part = _local_target(root, name + '.part')
    part.write_text(json.dumps(record, ensure_ascii=False), encoding='utf-8')
    os.replace(part, target)


def prepare_episode_tar(drive_tar, cache_root, *, reserve_bytes=2 * 1024 ** 3, progress=print):
    """Copy one uncompressed TAR; only publish the dataset after all NPZs are ready."""
    drive_tar = Path(drive_tar).resolve()
    stat = drive_tar.stat()
    if not drive_tar.is_file():
        raise ValueError('Drive TAR must be a file')
    identity = {'path': str(drive_tar), 'size': stat.st_size, 'mtime_ns': stat.st_mtime_ns}
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:24]
    root = Path(cache_root).resolve()
    root.mkdir(parents=True, exist_ok=True)
    dataset = _local_target(root, f'episodes_{key}')
    dataset.mkdir(exist_ok=True)
    ready_path = _local_target(dataset, '.tar_ready.json')
    ready = _read_cache_record(ready_path)

    # These checks touch only local files, never the 12,899 individual Drive NPZs.
    if ready.get('source') == identity and ready.get('files'):
        valid = all(_local_target(dataset, name).is_file()
                    and _local_target(dataset, name).stat().st_size == size
                    for name, size in ready['files'].items())
        manifest_path = _local_target(dataset, 'manifest.json')
        if valid and manifest_path.is_file():
            manifest_bytes = manifest_path.read_bytes()
            if hashlib.sha256(manifest_bytes).hexdigest() == ready.get('manifest_sha256'):
                progress(f'로컬 데이터 재사용: {len(ready["files"]):,}개 NPZ')
                return dataset, manifest_bytes

    local_tar = _local_target(root, f'{key}.tar')
    copy_record = _read_cache_record(_local_target(root, f'{key}.copy.json'))
    copied = (copy_record.get('source') == identity and local_tar.is_file()
              and local_tar.stat().st_size == stat.st_size)
    if not copied:
        # A plain TAR is approximately the size of its extracted files. Reserve both.
        needed = 2 * stat.st_size + reserve_bytes
        part = _local_target(root, f'{key}.tar.part')
        reclaimable = part.stat().st_size if part.is_file() else 0
        if shutil.disk_usage(root).free + reclaimable < needed:
            raise RuntimeError(f'로컬 공간 부족: TAR+해제본+여유 공간 약 {needed / 1024**3:.1f} GiB 필요')
        digest, copied_bytes, next_report = hashlib.sha256(), 0, 1024 ** 3
        progress(f'TAR 한 파일 복사 시작: {stat.st_size / 1024**3:.2f} GiB')
        with drive_tar.open('rb') as source, part.open('wb') as target:
            while chunk := source.read(16 * 1024 ** 2):
                target.write(chunk)
                digest.update(chunk)
                copied_bytes += len(chunk)
                if copied_bytes >= next_report:
                    progress(f'TAR 복사 {copied_bytes / stat.st_size:.0%}')
                    next_report += 1024 ** 3
        after = drive_tar.stat()
        if (copied_bytes != stat.st_size or (after.st_size, after.st_mtime_ns)
                != (stat.st_size, stat.st_mtime_ns)):
            raise RuntimeError('복사 중 TAR 원본이 변경되었거나 복사가 불완전합니다. 다시 실행하세요.')
        os.replace(part, local_tar)
        _write_cache_record(root, f'{key}.copy.json', {'source': identity, 'sha256': digest.hexdigest()})
    else:
        progress('완료된 로컬 TAR 복사본 재사용')

    # Manual extraction: no extractall, links, devices, absolute paths or traversal.
    with tarfile.open(local_tar, mode='r:') as archive:
        members, seen = {}, set()
        for member in archive:
            name = _archive_name(member.name)
            if name in seen:
                raise ValueError(f'Duplicate TAR member: {name}')
            seen.add(name)
            if not (member.isdir() or member.isfile()) or member.issparse():
                raise ValueError(f'TAR links/special/sparse files are not supported: {name}')
            if member.isfile():
                members[name] = member
        manifests = [name for name in members if PurePosixPath(name).name == 'manifest.json']
        if len(manifests) != 1:
            raise ValueError('TAR must contain exactly one manifest.json')
        manifest_name = manifests[0]
        if members[manifest_name].size > 64 * 1024 ** 2:
            raise ValueError('manifest.json exceeds 64 MiB')
        with archive.extractfile(members[manifest_name]) as stream:
            manifest_bytes = stream.read()
        manifest = json.loads(manifest_bytes)
        if not isinstance(manifest, dict) or not manifest.get('episodes'):
            raise ValueError('manifest에 에피소드가 없습니다.')
        prefix = PurePosixPath(manifest_name).parent
        expected = {}
        for entry in manifest['episodes']:
            relative = _archive_name(entry['file_path'])
            if not relative.endswith('.npz') or relative in expected:
                raise ValueError(f'Invalid/duplicate manifest NPZ: {relative}')
            member = members.get((prefix / relative).as_posix())
            if member is None or member.size <= 0:
                raise ValueError(f'Missing/empty NPZ in TAR: {relative}')
            if 'bytes' in entry and entry['bytes'] != member.size:
                raise ValueError(f'Manifest/TAR size mismatch: {relative}')
            expected[relative] = member
        pending = []
        for name, member in expected.items():
            target = _local_target(dataset, name)
            if not target.is_file() or target.stat().st_size != member.size:
                pending.append((name, member))
        needed = sum(member.size for _, member in pending) + len(manifest_bytes) + reserve_bytes
        if shutil.disk_usage(root).free < needed:
            raise RuntimeError(f'TAR 해제 공간 부족: 추가 {needed / 1024**3:.1f} GiB 필요')
        progress(f'로컬 TAR 해제: {len(pending):,}개, 재사용 {len(expected)-len(pending):,}개')
        for index, (name, member) in enumerate(pending, 1):
            target = _local_target(dataset, name)
            target.parent.mkdir(parents=True, exist_ok=True)
            part = _local_target(dataset, name + '.part')
            with archive.extractfile(member) as source, part.open('wb') as output:
                shutil.copyfileobj(source, output, length=16 * 1024 ** 2)
            if part.stat().st_size != member.size:
                raise RuntimeError(f'Incomplete extracted NPZ: {name}')
            os.replace(part, target)
            if index % 500 == 0 or index == len(pending):
                progress(f'로컬 해제 {index:,}/{len(pending):,}')

    manifest_part = _local_target(dataset, 'manifest.json.part')
    manifest_part.write_bytes(manifest_bytes)
    os.replace(manifest_part, _local_target(dataset, 'manifest.json'))
    _write_cache_record(dataset, '.tar_ready.json', {
        'source': identity, 'manifest_sha256': hashlib.sha256(manifest_bytes).hexdigest(),
        'files': {name: member.size for name, member in expected.items()},
    })
    progress(f'데이터 준비 완료: {dataset}')
    return dataset, manifest_bytes


In [ ]:
DRIVE_TAR_PATH = '/content/drive/MyDrive/ColabData/datasets/stockbot/20260811/extracted_episodes_entry_pattern.tar'  # @param {type:"string"}
LOCAL_CACHE_ROOT = Path('/content/episode_tar_cache')

LOCAL_DATA_DIR, manifest_bytes = prepare_episode_tar(DRIVE_TAR_PATH, LOCAL_CACHE_ROOT)
MANIFEST_SHA256 = hashlib.sha256(manifest_bytes).hexdigest()
manifest = json.loads(manifest_bytes)
print(f'에피소드 {len(manifest["episodes"]):,}개 → {LOCAL_DATA_DIR}')


In [ ]:
def validate_entry_pattern_dataset(config, manifest):
    from ai_trader.grpo.entry_pattern import DEFAULT_ENTRY_PATTERN, validate_entry_pattern
    if config.get('seq_len') != 2048:
        raise ValueError('이 노트북은 seq_len=2048 전용입니다. 이전 체크포인트는 원본 노트북에서 재개하세요.')
    active = validate_entry_pattern(config.get('entry_pattern_config'))
    required = {key: value for key, value in DEFAULT_ENTRY_PATTERN.items() if key != 'policy_coef'}
    if active is None or any(active[key] != value for key, value in required.items()):
        raise ValueError('30억 원/1%/5~60초 진입 및 체결 후 1~5초 학습 설정이 필요합니다. 새 학습을 사용하세요.')
    if config.get('execution_action_mask') is not True:
        raise ValueError('매수 조건을 적용하려면 execution_action_mask=True가 필요합니다.')
    metadata = manifest.get('metadata', {})
    extracted = validate_entry_pattern(metadata.get('entry_pattern_config'))
    selection = ('min_buy_notional_krw', 'min_price_return', 'min_window_seconds', 'max_window_seconds')
    if extracted is None or any(extracted[key] != active[key] for key in selection):
        raise ValueError('이전 NPZ 또는 다른 진입 조건의 TAR입니다. 4번 안내대로 재추출하세요.')
    source = metadata.get('extraction', {}).get('buy_notional_source')
    if source not in ('signed_volume_positive_buy', 'cumulative_buy_notional'):
        raise ValueError('매수 방향 체결금액의 추출 출처가 없습니다. 수정된 추출기로 재추출하세요.')
    if metadata.get('expected_features') != 29 or config.get('features') != 29:
        raise ValueError('시초가를 포함한 29개 시장 특징이 필요합니다.')
    episodes = manifest.get('episodes', [])
    usable = [ep for ep in episodes if any(
        end >= 2047 and start < int(ep['length']) - 1
        for start, end in ep.get('entry_signal_ranges', []))]
    if not usable:
        raise ValueError('2048행 관측 이후에 진입 조건을 만족하는 에피소드가 없습니다.')
    return {'total_episodes': len(episodes), 'qualifying_episodes': len(usable),
            'buy_notional_source': source, 'seq_len': 2048, 'entry_pattern_config': active}


## 5. A100 프로필 및 실험 설정

| 항목 | A100 40GB | A100 80GB |
|---|---:|---:|
| 관측 시장 행 / 정책 판단 상한 | 2,048 / 300 | 2,048 / 300 |
| CNN / mLSTM / FC 차원 | 256 / 512 / 512 | 256 / 512 / 512 |
| 학습 미니배치 시작값 | 16 | 32 |
| 회당 에피소드 / 그룹 | 16 / 4 | 16 / 4 |
| gradient checkpoint 분할 / update epoch | 16 / 2 | 16 / 2 |
| worker 상한 | 4 | 8 |

GPU 여유 메모리·CPU·RAM에 따라 축소합니다. 두 GPU에서 관측·모델 크기는 유지합니다.
FP32 rollout 한 벌은 기본 16개 에피소드의 관측 배열만 약 **2.38 GiB**이며 결합 배열, worker 캐시 및 프로세스 메모리가 추가로 필요합니다.
여유 RAM이 16 GiB보다 작으면 회당 8개 에피소드를 사용합니다.

`MODE='new'`가 기본입니다. `resume`은 저장된 관측·학습·체결 설정과 optimizer/진행률을 복원합니다.
`finetune`은 호환되는 설정과 가중치를 읽고 optimizer/진행률을 새로 시작합니다.
`TOTAL_TIMESTEPS`는 재개 시 **누적 목표치**이므로 저장된 값보다 크게 지정하세요.
새 학습/미세조정에는 비어 있는 `RUN_NAME`을 사용하세요.

수수료·세금과 지연·스프레드 값은 예시 시뮬레이션 가정입니다. 사용하는 시장/계좌/데이터에 맞춰 조정하세요.
`DIAGNOSTIC_CHECKPOINT`에는 같은 관측·모델 설정의 기존 체크포인트 경로를 선택적으로 넣습니다. `LOAD_POLICY`와 별개이며, 신규 학습의 시작 가중치를 바꾸지 않습니다. `policy_update_checks`, `rollout_logprob_tolerance`, `kl_probe_samples`, `no_trade_max_validations`는 아래 설정 셀에서 바꾼 뒤 6→7→8번 셀을 다시 실행하세요. resume/finetune은 누락된 검사 설정을 비활성 상태로 복원합니다. 재개 시 검사를 켜려면 `restore_run_config` 호출 이후, 메모리 추정 전에 CONFIG를 명시적으로 갱신하세요.

`ENABLE_TF32`는 새 학습과 재개 모두 현재 체크박스 선택을 사용합니다. 저장된 값은 비교 안내만 출력하며 현재 선택을 덮어쓰지 않습니다. TF32를 바꾼 뒤에는 7번 점검을 다시 실행하세요. TF32 영향은 같은 실패 자료의 비교 검사로 확인합니다.

`MODE=resume`에서 LR을 바꾸려면 `RESUME_LR`에 양수를 지정합니다. 0이면 체크포인트의 Adam LR을 유지합니다. `LEARNING_RATE`는 new/finetune용이며, 재개 LR 변경은 Adam 모멘트를 초기화하지 않습니다. 다음 체크포인트에도 실제 적용 LR이 기록됩니다.


In [ ]:
def model_obs_dim(config):
    from lib.observations import ACCOUNT_FIELDS, EXECUTION_FIELDS
    return (config['features'] + 15
            + (len(ACCOUNT_FIELDS) if config.get('account_observations', False) else 0)
            + (len(EXECUTION_FIELDS) if config.get('execution_observations', False) else 0))

def estimated_host_gib(config, episode_rows=None):
    states = (config['episodes_per_group'] * config['num_groups'] * config['episode_steps']
              * config['seq_len'] * model_obs_dim(config) * 4 / GiB)
    # 검증 환경은 같은 프로세스의 원시 캐시를 공유하고 경로 배열만 따로 보유합니다.
    rows = (episode_rows if episode_rows is not None else
            config['seq_len'] + config['episode_steps'] + config.get('liquidation_max_steps', 0))
    eval_arrays = rows * (config['features'] * 4 + 44 * 8) + config['seq_len'] * model_obs_dim(config) * 4
    extra_eval = max(0, config.get('evaluation_workers', 1) - 1) * 4 * eval_arrays / GiB
    timed_replay = (config['num_workers'] + config.get('evaluation_workers', 1)) * 4 * rows * (config['features'] * 4 + 44 * 8) / GiB if episode_rows is not None else 0
    return timed_replay + 3 * states + config['num_workers'] * (0.75 + config['cache_max_bytes'] / GiB) + 2 + extra_eval

def a100_profile(vram_gib, free_vram_gib, ram_gib, cpu_count):
    if vram_gib < 30 or free_vram_gib < 4 or ram_gib < 6:
        raise ValueError('A100 프로필을 시작할 여유 GPU/RAM이 부족합니다.')
    large = vram_gib >= 70
    worker_cap = 8 if large else 4
    ram_workers = 8 if ram_gib >= 48 else (4 if ram_gib >= 24 else 2)
    profile = dict(seq_len=2048, features=29, episode_steps=300, account_observations=True, execution_observations=True,
                   cnn_channels=256, rnn_hidden_dim=512, hidden_dim=512,
                   batch_size=32 if large else 16, checkpoint_segments=16,
                   episodes_per_group=4 if ram_gib >= 16 else 2, num_groups=4,
                   num_workers=max(1, min(worker_cap, ram_workers, max(1, cpu_count - 1))),
                   cache_max_bytes=128 * 1024 ** 2)
    if free_vram_gib < 16:
        profile['batch_size'] = min(profile['batch_size'], 8)
    while estimated_host_gib(profile) > 0.70 * ram_gib:
        if profile['num_workers'] > 1:
            profile['num_workers'] //= 2
        elif profile['episodes_per_group'] > 2:
            profile['episodes_per_group'] = 2
        else:
            raise ValueError('롤아웃 저장 공간 부족: 다른 메모리 사용을 줄이거나 고용량 RAM 런타임을 선택하세요.')
    return profile

def restore_run_config(base, checkpoint, mode):
    if mode not in ('resume', 'finetune'):
        raise ValueError('mode must be resume or finetune')
    ObservationBuilder.from_schema(checkpoint.get('observation_schema'))
    saved = checkpoint.get('extra_state', {}).get('training_config')
    if not saved or not checkpoint.get('extra_state', {}).get('date_splits'):
        raise ValueError('학습 설정/날짜 이력이 없는 구형 체크포인트입니다. 신규 학습하세요.')
    merged = dict(saved)
    # Invocation requests are never inherited from the old checkpoint.
    merged['resume_lr'] = base.get('resume_lr') if mode == 'resume' else None
    merged['capture_update_bundle'] = None
    merged.setdefault('max_stages', checkpoint['observation_schema']['max_stages'])
    merged.setdefault('selection_require_liquidation', True)
    merged.setdefault('execution_action_mask', False)
    merged.setdefault('entry_pattern_config', None)
    merged.setdefault('no_trade_patience', 0)
    merged.setdefault('no_trade_max_validations', 0)
    merged.setdefault('policy_update_checks', False)
    merged.setdefault('rollout_logprob_tolerance', 1e-3)
    merged.setdefault('kl_probe_samples', 32)
    merged.setdefault('profitable_min_round_trips', 20)
    merged.setdefault('profitable_min_traded_dates', 3)
    merged['account_observations'] = checkpoint['observation_schema']['version'] in (3, 4)
    merged['execution_observations'] = checkpoint['observation_schema']['version'] == 4
    merged.setdefault('decision_interval_seconds', 0.0)
    merged.setdefault('episode_duration_seconds', 0.0)
    merged.setdefault('group_advantage_coef', 1.0)
    merged.setdefault('training_seed', base.get('training_seed', 42))
    merged.setdefault('liquidation_max_steps', 0)
    merged.setdefault('lambda_gae', 0.95)
    for key in ('evaluation_workers', 'diagnostics_interval', 'diagnostics_max_samples'):
        merged[key] = base[key]
    # 세션 자원과 파일 위치만 새 값 사용. 거래/날짜/관측/optimizer 관련 설정은 보존.
    for key in ('extracted_dir', 'output_dir', 'device', 'load_policy', 'total_timesteps',
                'num_workers', 'batch_size', 'cache_max_bytes', 'checkpoint_segments'):
        merged[key] = base[key]
    merged['resume'] = mode == 'resume'
    if merged['resume'] and not all(key in checkpoint for key in
            ('optimizer_state_dict', 'total_timesteps', 'num_updates', 'iteration')):
        raise ValueError('resume에는 optimizer와 진행률이 포함된 전체 학습 체크포인트가 필요합니다.')
    if merged['resume'] and merged['total_timesteps'] <= checkpoint['total_timesteps']:
        raise ValueError('TOTAL_TIMESTEPS를 체크포인트의 누적 timesteps보다 크게 지정하세요.')
    if mode == 'finetune':
        merged['lr'] = base['lr']
    elif merged['resume_lr'] is not None:
        merged['lr'] = merged['resume_lr']
    return merged

In [ ]:
import math

MODE = 'new'  # @param ["new", "resume", "finetune"]
RUN_NAME = 'scalping_entry_pattern_2048_run01'  # @param {type:"string"}
EXPERIMENT = 'timed_gae_only'  # @param ["cost_observations", "timed_decisions", "gae_only", "long_credit", "monte_carlo_credit", "timed_gae_only", "timed_long_credit", "timed_monte_carlo_credit"]
LOAD_POLICY = ''  # @param {type:"string"}
DIAGNOSTIC_CHECKPOINT = ''  # @param {type:"string"}
TOTAL_TIMESTEPS = 1_000_000  # @param {type:"integer"}
LEARNING_RATE = 3e-5  # @param {type:"number"}
RESUME_LR = 0.0  # @param {type:"number"}
MAX_STAGES = 1  # @param {type:"integer"}
REQUIRE_ORDER_BOOK = True  # @param {type:"boolean"}
ENABLE_TF32 = True  # @param {type:"boolean"}
SEED = 42  # @param {type:"integer"}

if MODE not in ('new', 'resume', 'finetune') or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ('', '.', '..'):
    raise ValueError('MODE와 단일 폴더 이름 RUN_NAME을 확인하세요.')
if isinstance(RESUME_LR, bool) or not isinstance(RESUME_LR, (int, float)) or not math.isfinite(RESUME_LR) or RESUME_LR < 0:
    raise ValueError('RESUME_LR은 0 또는 유한한 양수여야 합니다. 0은 저장된 LR을 유지합니다.')
if MODE != 'resume' and RESUME_LR != 0:
    raise ValueError('RESUME_LR은 MODE=resume에서만 사용합니다.')
OUTPUT_DIR = Path('/content/drive/MyDrive/ColabData/stockbot/models') / RUN_NAME
if MODE != 'resume' and OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError('새 학습/미세조정은 비어 있는 RUN_NAME을 지정하세요.')
profile = a100_profile(VRAM_GIB, torch.cuda.mem_get_info()[0] / GiB, available_ram_gib(), CPU_COUNT)
CONFIG = dict(TrainingConfig().__dict__)
CONFIG.update(profile)
CONFIG.update(extracted_dir=str(LOCAL_DATA_DIR), output_dir=str(OUTPUT_DIR), device='cuda',
              total_timesteps=TOTAL_TIMESTEPS, lr=LEARNING_RATE, gamma=1.0, clip=0.1, kl_target=0.01,
              entropy_coef=0.01, value_coef=0.5, max_grad_norm=0.5, num_epochs=2, use_gae=True,
              rolling_window_size=512, rolling_min_samples=64, use_raw_data=True,
              evaluation_episodes=64, evaluation_interval=5, evaluation_seed=SEED, training_seed=SEED,
              evaluation_workers=min(8, profile['num_workers']),
              account_observations=True, execution_observations=True, liquidation_max_steps=300, lambda_gae=0.95,
              decision_interval_seconds=0.0, episode_duration_seconds=0.0, group_advantage_coef=1.0,
              diagnostics_interval=5, diagnostics_max_samples=256,
              selection_require_liquidation=True,
              entry_pattern_config=dict(min_buy_notional_krw=3_000_000_000.0, min_price_return=0.01,
                                        min_window_seconds=5.0, max_window_seconds=60.0,
                                        target_min_seconds=1.0, target_max_seconds=5.0, policy_coef=1.0),
              execution_action_mask=True, no_trade_patience=3, no_trade_max_validations=4,
              policy_update_checks=True, rollout_logprob_tolerance=1e-3, kl_probe_samples=32,
              profitable_min_round_trips=20, profitable_min_traded_dates=3,
              validation_fraction=0.2, test_fraction=0.2, embargo_dates=0,
              train_end_date=None, validation_end_date=None, checkpoint_interval=1,
              initial_cash=1_000_000.0, max_stages=MAX_STAGES, max_holding_seconds=300.0, stop_loss_pct=2.0,
              transaction_cost_rate=0.00015, buy_tax_rate=0.0, sell_tax_rate=0.0018,
              no_trade_penalty=0.0, win_bonus=0.0, loss_penalty=0.0, buy_signal_bonus=0.0,
              step_reward_scale=1.0, max_trades_per_episode=None, revert_patience=0,
              load_policy=LOAD_POLICY or None, resume=False,
              resume_lr=float(RESUME_LR) if RESUME_LR else None, capture_update_bundle=None,
              execution_config=dict(order_latency_ms=100, cancel_latency_ms=50,
                                    order_ttl_seconds=2.0, spread_bps=10, slippage_bps=2,
                                    fallback_depth=100, require_order_book=REQUIRE_ORDER_BOOK,
                                    max_quote_age_seconds=1.0, tick_size=0.0))
# 관측 개선을 기준으로 한 번에 한 실험을 비교합니다. 값들은 최적값이 아닌 대조 실험 후보입니다.
EXPERIMENTS = {
    'cost_observations': {},
    'timed_decisions': dict(decision_interval_seconds=1.0, episode_duration_seconds=300.0),
    'gae_only': dict(group_advantage_coef=0.0),
    'long_credit': dict(lambda_gae=0.99),
    'monte_carlo_credit': dict(lambda_gae=1.0),
    'timed_gae_only': dict(decision_interval_seconds=1.0, episode_duration_seconds=300.0, group_advantage_coef=0.0, lambda_gae=0.95),
    'timed_long_credit': dict(decision_interval_seconds=1.0, episode_duration_seconds=300.0, group_advantage_coef=0.0, lambda_gae=0.99),
    'timed_monte_carlo_credit': dict(decision_interval_seconds=1.0, episode_duration_seconds=300.0, group_advantage_coef=0.0, lambda_gae=1.0),
}
if EXPERIMENT not in EXPERIMENTS:
    raise ValueError(f'지원하지 않는 EXPERIMENT={EXPERIMENT!r}. 지원 목록: {list(EXPERIMENTS)}. 노트북 설정 셀도 최신으로 갱신하세요.')
if MODE == 'new':
    CONFIG.update(EXPERIMENTS[EXPERIMENT])
else:
    print('resume/finetune은 체크포인트의 관측·시간·학습 설정을 복원합니다. 비교 실험에는 new를 사용하세요.')
loaded_checkpoint = None
if MODE == 'new' and LOAD_POLICY:
    raise ValueError('LOAD_POLICY를 비우거나 MODE를 resume/finetune으로 바꾸세요.')
if MODE != 'new':
    if not LOAD_POLICY or not Path(LOAD_POLICY).is_file():
        raise FileNotFoundError('재개/미세조정할 체크포인트 경로를 지정하세요.')
    if (MODE == 'resume' and OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir())
            and not Path(LOAD_POLICY).resolve().is_relative_to(OUTPUT_DIR.resolve())):
        raise ValueError('다른 실험의 출력에 덮어쓸 수 없습니다. 해당 폴더의 체크포인트 또는 새 RUN_NAME을 사용하세요.')
    # 직접 생성한 신뢰할 수 있는 체크포인트만 사용하세요.
    loaded_checkpoint = torch.load(LOAD_POLICY, map_location='cpu', weights_only=True)
    CONFIG = restore_run_config(CONFIG, loaded_checkpoint, MODE)
    if MODE == 'resume':
        prior_info = Path(LOAD_POLICY).parent.parent / 'colab_run.json'
        if prior_info.exists():
            prior = json.loads(prior_info.read_text())
            if prior['manifest_sha256'] != MANIFEST_SHA256:
                raise ValueError('재개할 실험의 데이터 manifest가 달라졌습니다.')
            print(f"재개 TF32: 저장값={prior.get('enable_tf32')!r}, 현재 선택={ENABLE_TF32!r} (현재 선택 사용)")
            SEED = prior['seed']
    print('체크포인트 설정 복원 완료. finetune의 학습률만 현재 LR 설정을 사용합니다.')

ENTRY_DATA_SUMMARY = validate_entry_pattern_dataset(CONFIG, manifest)
print('진입 조건 데이터 검사:', ENTRY_DATA_SUMMARY)

timed_replay = CONFIG['decision_interval_seconds'] > 0 or CONFIG['episode_duration_seconds'] > 0
max_replay_rows = max(int(ep['length']) for ep in manifest['episodes']) if timed_replay else None
host_estimate = estimated_host_gib(CONFIG, max_replay_rows)
if host_estimate > 0.70 * available_ram_gib():
    raise MemoryError(f'저장된 롤아웃 설정은 현재 RAM 예산을 넘습니다: 추정 {host_estimate:.1f} GiB')
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print(f'호스트 학습 메모리 추정 {host_estimate:.1f} GiB (상한 보장 아님)')

## 6. 날짜 분리 및 진입 조건·관측·체결 검사

거래일을 train/validation/test로 분리합니다. 각 분할에 2048행 관측 이후 진입 후보가 있는지 확인하고 **훈련 분할만** 열어 `(2048, 65)` 관측과 체결 환경을 점검합니다. 검증·테스트의 가격을 사용해 설정을 조정하지 않습니다.


In [ ]:
import numpy as np
from ai_trader.grpo.evaluation import normalize_date, validate_checkpoint_dates

unknown = set(CONFIG) - set(TrainingConfig().__dict__)
if unknown:
    raise ValueError(f'현재 trainer에서 지원하지 않는 설정: {unknown}')
training_config = TrainingConfig()
for key, value in CONFIG.items():
    setattr(training_config, key, value)
training_config.validate()
DATE_SPLITS = prepare_date_splits(training_config)
for partition, dates in DATE_SPLITS.items():
    lengths = [int(ep['length']) for ep in manifest['episodes'] if normalize_date(ep['date']) in dates]
    timed = CONFIG['decision_interval_seconds'] > 0 or CONFIG['episode_duration_seconds'] > 0
    minimum_rows = (CONFIG['seq_len'] + 1 if timed else
                    CONFIG['seq_len'] + CONFIG['episode_steps'] + CONFIG['liquidation_max_steps'])
    candidates = [ep for ep in manifest['episodes'] if normalize_date(ep['date']) in dates
                  and int(ep['length']) >= minimum_rows
                  and any(end >= CONFIG['seq_len'] - 1 and start < int(ep['length']) - 1
                          for start, end in ep.get('entry_signal_ranges', []))]
    full = len(candidates)
    if not full:
        raise ValueError(f'{partition}: 관측 이력과 진입 조건을 만족하는 에피소드가 없습니다.')
    print(f'{partition}: {dates[0]} ~ {dates[-1]}, {len(dates)}일, 진입 후보가 있는 에피소드 {full}/{len(lengths)}개')
if loaded_checkpoint is not None:
    validate_checkpoint_dates(loaded_checkpoint, DATE_SPLITS)
probe_env = create_environment(training_config, 'cpu', DATE_SPLITS['train'])
try:
    probe_observation, probe_info = probe_env.reset(seed=SEED)
    probe_action_mask = probe_env.action_masks() if CONFIG.get('execution_action_mask', False) else None
    if loaded_checkpoint is not None:
        probe_env.observation_builder.validate_schema(loaded_checkpoint['observation_schema'])
    assert probe_observation.shape == (CONFIG['seq_len'], model_obs_dim(CONFIG))
    assert np.isfinite(probe_observation).all()
    print('관측:', probe_observation.shape, '| 가격 단위:', probe_env.price_unit)
    print('정책 판단 간격(초):', CONFIG['decision_interval_seconds'],
          '| 에피소드 시간 상한(초):', CONFIG['episode_duration_seconds'])
    print('샘플 시장 재생 길이(초):', float(probe_env.timestamps[-1] - probe_env.current_time_seconds),
          '| 정책 판단 상한:', CONFIG['episode_steps'])
    print('체결 모델:', probe_info.get('execution_model', probe_env.simulator.summary()['execution_model']))
    OBSERVATION_SCHEMA = probe_env.observation_schema
finally:
    probe_env.close()
    type(probe_env).clear_episode_cache()
    del probe_env
loaded_checkpoint = None
gc.collect()

## 7. GPU 배치 사전 점검

현재 모델의 forward/backward와 Adam 갱신을 실행해 메모리를 측정합니다.
OOM 또는 GPU 여유 공간의 70%를 초과하면 미니배치를 절반으로 줄입니다.
이 검사는 전체 데이터 학습의 최대 메모리를 보장하지 않으므로 실제 학습 로그도 확인하세요.

[TF32](https://docs.pytorch.org/docs/stable/notes/cuda.html)는 FP32 텐서를 유지하면서 내부 행렬 연산 정밀도를 낮춥니다.
동일 설정을 **실제 학습 자식 프로세스에도 적용**합니다. 현재 mLSTM의 AMP/BF16 학습은 사용하지 않습니다.
실제 환경에서 여러 관측과 실행 마스크를 수집하고, worker 크기의 무기울기 rollout과 미니배치 크기의 기울기 계산 경로에서 같은 행동의 log probability를 비교합니다. fresh 검사는 0이 아닌 정책 head로 실행하며, `DIAGNOSTIC_CHECKPOINT`를 지정하면 그 가중치도 그대로 별도로 검사합니다. 최대 오차가 `rollout_logprob_tolerance`를 넘으면 학습을 시작하지 않습니다. 실패 후 허용 오차만 넓히지 말고 보고된 차이를 확인하세요.


In [ ]:
from ai_trader.grpo.policies.scalping_policy_xlstm import GRPOPolicyE2EXLSTM

# 학습 프로세스에서 재사용할 코드. PyTorch 2.9+와 이전 설정 API를 섞지 않습니다.
GPU_SETUP = """
import os, random, json, numpy as np, torch
from ai_trader.grpo.runtime_precision import set_tf32, precision_metadata
enable_tf32 = os.environ.get('SCALPING_TF32', '1') == '1'
set_tf32(enable_tf32)
torch.backends.cudnn.benchmark = False
torch.set_num_threads(1)
seed = int(os.environ.get('SCALPING_SEED', '42'))
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
print('Runtime precision:', json.dumps(precision_metadata(), sort_keys=True))
"""
os.environ['SCALPING_TF32'] = '1' if ENABLE_TF32 else '0'
os.environ['SCALPING_SEED'] = str(SEED)
exec(GPU_SETUP)

def probe_batch(config, observation, action_mask=None):
    policy = GRPOPolicyE2EXLSTM(obs_dim=observation.shape[-1],
        cnn_channels=config['cnn_channels'], rnn_hidden_dim=config['rnn_hidden_dim'],
        fc_hidden_dim=config['hidden_dim'], max_stages=config['max_stages'], checkpoint_segments=config['checkpoint_segments'],
        execution_action_mask=config.get('execution_action_mask', False)).cuda()
    if config['load_policy']:
        checkpoint = torch.load(config['load_policy'], map_location='cpu', weights_only=True)
        policy.load_state_dict(checkpoint['policy_state_dict'], strict=True)
        del checkpoint
    optimizer = torch.optim.Adam(policy.parameters(), lr=config['lr'])
    batch = torch.as_tensor(observation, device='cuda').unsqueeze(0).repeat(config['batch_size'], 1, 1)
    if config.get('execution_action_mask', False) and action_mask is None:
        raise ValueError('GPU probe requires the executable action mask from its observation')
    mask_args = ({'action_masks': torch.as_tensor(action_mask, dtype=torch.bool, device='cuda').unsqueeze(0).repeat(config['batch_size'], 1)}
                 if action_mask is not None else {})
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    for _ in range(2):
        policy.eval()
        with torch.no_grad():
            actions, _ = policy.get_action(batch, **mask_args)
        policy.train()
        log_probs, entropy, values = policy.evaluate_actions(batch, actions, **mask_args)
        loss = -log_probs.mean() - config['entropy_coef'] * entropy.mean() + config['value_coef'] * values.square().mean()
        if not torch.isfinite(loss):
            raise FloatingPointError('사전 점검 loss에 NaN/Inf가 있습니다.')
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        norm = torch.nn.utils.clip_grad_norm_(policy.parameters(), config['max_grad_norm'], error_if_nonfinite=True)
        optimizer.step()
    torch.cuda.synchronize()
    return {'batch_size': config['batch_size'], 'peak_gib': torch.cuda.max_memory_allocated() / GiB,
            'two_steps_seconds': time.perf_counter() - started, 'grad_norm': float(norm)}

GPU_PROBE = None
while CONFIG['batch_size'] >= 8:
    gc.collect()
    torch.cuda.empty_cache()
    memory_budget = 0.70 * torch.cuda.mem_get_info()[0] / GiB
    try:
        result = probe_batch(CONFIG, probe_observation, probe_action_mask)
        print(result)
        if result['peak_gib'] <= memory_budget:
            GPU_PROBE = result
            break
    except torch.cuda.OutOfMemoryError:
        print(f'배치 {CONFIG["batch_size"]}: GPU 메모리 부족')
    CONFIG['batch_size'] //= 2
gc.collect()
torch.cuda.empty_cache()
if GPU_PROBE is None:
    raise RuntimeError('사전 점검 실패. GPU 메모리를 비우고 관측/모델 설정을 확인하세요.')
def collect_likelihood_observations(config, dates, sample_count, seed):
    """Use real market/position states, with a fresh seed for each short path."""
    check_config = TrainingConfig()
    for key, value in config.items():
        setattr(check_config, key, value)
    check_env = create_environment(check_config, 'cpu', dates)
    states, masks = [], []
    path_length = max(2, int(config['num_workers']))
    try:
        observation, info = check_env.reset(seed=seed)
        for index in range(sample_count):
            mask = np.asarray(check_env.action_masks(), dtype=bool)
            states.append(np.asarray(observation, dtype=np.float32).copy())
            masks.append(mask.copy())
            # Exercise entry, waiting, and exit states without bypassing order masks.
            desired_action = (1, 0, 0, 2)[index % 4]
            action = desired_action if mask[desired_action] else 0
            observation, _, terminated, truncated, info = check_env.step(action)
            if index + 1 < sample_count and (terminated or truncated or (index + 1) % path_length == 0):
                observation, info = check_env.reset(seed=seed + index + 1)
        return np.stack(states), np.stack(masks)
    finally:
        check_env.close()
        type(check_env).clear_episode_cache()


def check_policy_likelihood_batches(policy, states, masks, config, device='cuda'):
    """Compare frozen rollout batches with gradient-enabled update batches."""
    from ai_trader.grpo.policy_update_checks import check_rollout_likelihood
    was_training = policy.training
    actions, old_log_probs = [], []
    rollout_batch_size = int(config['num_workers'])
    use_masks = bool(config.get('execution_action_mask', False))
    try:
        policy.eval()
        with torch.no_grad():
            for start in range(0, len(states), rollout_batch_size):
                stop = start + rollout_batch_size
                batch = torch.as_tensor(states[start:stop], device=device)
                kwargs = ({'action_masks': torch.as_tensor(masks[start:stop], dtype=torch.bool, device=device)}
                          if use_masks else {})
                action, log_prob, _ = policy.get_action_with_value(batch, deterministic=False, **kwargs)
                actions.append(action.detach().cpu())
                old_log_probs.append(log_prob.detach().cpu())
        result = check_rollout_likelihood(
            policy, states, torch.cat(actions), torch.cat(old_log_probs),
            masks if use_masks else None, batch_size=int(config['batch_size']),
            tolerance=float(config['rollout_logprob_tolerance']), device=device)
        # The full action reference is used by the trainer's KL checks, not JSON logs.
        result.pop('reference_log_probs', None)
        return {**result, 'rollout_batch_size': rollout_batch_size,
                'update_batch_size': int(config['batch_size']),
                'seq_len': int(states.shape[1]), 'obs_dim': int(states.shape[2])}
    finally:
        policy.train(was_training)


def run_likelihood_preflight(config, states, masks, checkpoint_path='', device='cuda'):
    """Diagnostic weights are independent from the weights selected for training."""
    checkpoint = None
    if checkpoint_path:
        if not Path(checkpoint_path).is_file():
            raise FileNotFoundError(f'진단 체크포인트가 없습니다: {checkpoint_path}')
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
        ObservationBuilder.from_schema(OBSERVATION_SCHEMA).validate_schema(checkpoint['observation_schema'])
        saved = checkpoint.get('extra_state', {}).get('training_config', {})
        for name in ('cnn_channels', 'rnn_hidden_dim', 'hidden_dim', 'max_stages'):
            if name not in saved or saved[name] != config[name]:
                raise ValueError(f'진단 체크포인트와 현재 모델 설정이 다릅니다: {name}')
        if bool(saved.get('execution_action_mask', False)) != bool(config.get('execution_action_mask', False)):
            raise ValueError('진단 체크포인트의 execution_action_mask 설정이 다릅니다.')
    policy = GRPOPolicyE2EXLSTM(obs_dim=states.shape[-1],
        cnn_channels=config['cnn_channels'], rnn_hidden_dim=config['rnn_hidden_dim'],
        fc_hidden_dim=config['hidden_dim'], max_stages=config['max_stages'],
        checkpoint_segments=config['checkpoint_segments'],
        execution_action_mask=config.get('execution_action_mask', False)).to(device)
    try:
        if checkpoint is not None:
            policy.load_state_dict(checkpoint['policy_state_dict'], strict=True)
        else:
            # A zero actor head would hide differences in the recurrent features.
            with torch.no_grad():
                torch.nn.init.normal_(policy.policy_head.weight, std=0.02)
                policy.policy_head.bias.copy_(torch.tensor([-.2, .1, .05], device=device))
        result = check_policy_likelihood_batches(policy, states, masks, config, device=device)
        result['source'] = checkpoint_path or 'fresh_nonzero_policy_head'
        result['policy_head_nonzero'] = bool(torch.count_nonzero(policy.policy_head.weight).item())
        return result
    finally:
        del policy, checkpoint
        gc.collect()
        if str(device).startswith('cuda'):
            torch.cuda.empty_cache()


def likelihood_probe_fingerprint(config, diagnostic_checkpoint, enable_tf32):
    from ai_trader.grpo.runtime_precision import precision_metadata
    return json.dumps(dict(config=config, diagnostic_checkpoint=diagnostic_checkpoint,
                           enable_tf32=bool(enable_tf32), precision=precision_metadata()), sort_keys=True)


POLICY_LIKELIHOOD_CHECKS = []
PROBED_LIKELIHOOD_SETTINGS = None
if CONFIG.get('policy_update_checks', False):
    sample_count = max(CONFIG['batch_size'], CONFIG['num_workers'], CONFIG['kl_probe_samples'])
    check_states, check_masks = collect_likelihood_observations(CONFIG, DATE_SPLITS['train'], sample_count, SEED)
    try:
        # Test a nonconstant fresh head even when checking trained weights as well.
        check_sources = ['']
        for source in (CONFIG.get('load_policy') or '', DIAGNOSTIC_CHECKPOINT):
            if source and source not in check_sources:
                check_sources.append(source)
        for source in check_sources:
            result = run_likelihood_preflight(CONFIG, check_states, check_masks, checkpoint_path=source)
            POLICY_LIKELIHOOD_CHECKS.append(result)
            print('Rollout/update likelihood 검사 통과:', json.dumps(result, ensure_ascii=False))
    finally:
        del check_states, check_masks
        gc.collect()
        torch.cuda.empty_cache()
elif DIAGNOSTIC_CHECKPOINT:
    raise ValueError('DIAGNOSTIC_CHECKPOINT 검사에는 policy_update_checks=True가 필요합니다.')
PROBED_LIKELIHOOD_SETTINGS = likelihood_probe_fingerprint(CONFIG, DIAGNOSTIC_CHECKPOINT, ENABLE_TF32)

PROBED_CONFIG = json.dumps(CONFIG, sort_keys=True)
print('확정 미니배치:', CONFIG['batch_size'])

## 7-1. 실패한 likelihood 배치 재현 (선택)

학습 실패 로그에 출력된 bundle 경로를 넣습니다. 7번 GPU 점검 통과와 무관하게 독립 실행할 수 있습니다.
저장된 같은 가중치·관측·행동·마스크로 TF32 ON/OFF와 두 배치 크기를 비교합니다.
이는 저장한 표본의 재배치 진단이며 당시 전체 rollout 배치를 그대로 복원한 검사는 아닙니다.
학습 설정이나 허용 오차를 바꾸지 않고 별도 JSON 보고서를 저장합니다.


In [ ]:
LIKELIHOOD_FAILURE_BUNDLE = ''  # @param {type:"string"}
DIAGNOSTIC_OUTPUT = ''  # @param {type:"string"}
DIAGNOSTIC_DEVICE = 'auto'  # @param ["auto", "cpu", "cuda"]

if LIKELIHOOD_FAILURE_BUNDLE:
    bundle_path = Path(LIKELIHOOD_FAILURE_BUNDLE).expanduser()
    if not bundle_path.is_file():
        raise FileNotFoundError(f'실패 bundle이 없습니다: {bundle_path}')
    diagnostic_path = (Path(DIAGNOSTIC_OUTPUT).expanduser() if DIAGNOSTIC_OUTPUT else
                       bundle_path.with_name(bundle_path.stem + '_tf32_replay_' + str(time.time_ns()) + '.json'))
    replay_command = [sys.executable, '-u', '-m', 'ai_trader.grpo.diagnose_likelihood',
                      '--bundle', str(bundle_path), '--output', str(diagnostic_path),
                      '--device', DIAGNOSTIC_DEVICE]
    replay_result = subprocess.run(replay_command, cwd=REPO_PATH, env=dict(os.environ),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8')
    print(replay_result.stdout)
    if replay_result.returncode:
        raise subprocess.CalledProcessError(replay_result.returncode, replay_command, output=replay_result.stdout)
    print('TF32/배치 비교 보고서:', diagnostic_path)
    print(json.dumps(json.loads(diagnostic_path.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
else:
    print('선택 검사: LIKELIHOOD_FAILURE_BUNDLE을 지정하면 실행합니다.')


## 7-2. 같은 rollout과 Adam 상태로 학습률 비교 (선택)

`UPDATE_DIAGNOSTIC_CHECKPOINT`에 실제 학습 체크포인트(예: 19회차)를 지정하면 실행합니다.
저장된 정책과 Adam을 복원하고 FP32로 다음 한 그룹의 실제 rollout(기본 16개)을 한 번 수집합니다.
이 자료를 고정하여 LR 3e-5 / 1e-5 / 3e-6을 각각 같은 상태에서 비교합니다.
LR 비교 전에 BUY 결정과 실제 주문 손익·GAE를 연결한 `_entry_credit.json`도 자동 저장합니다. 수익 거래의 음수 advantage, 비용별 손익, 미체결·미청산을 구분하며 새 하이퍼파라미터는 필요 없습니다.
검증·테스트 평가와 기존 학습 체크포인트 저장은 실행하지 않으며 7번 GPU 점검 통과와 무관하게 실행할 수 있습니다.
경로를 비우면 큰 bundle은 /content/stockbot_update_diagnostics에, JSON 보고서는 Drive의 체크포인트 실행 폴더 diagnostics에 새 이름으로 저장합니다. 직접 지정할 때는 기존 파일과 겹치지 않아야 합니다.
결과는 한 번의 업데이트 진단이며 수익률 검증이 아닙니다. 정상 학습은 8번에서 별도로 시작합니다.

캡처와 비교는 수 분 이상 걸릴 수 있으며 진행 로그를 즉시 표시합니다. Drive 파일시스템이 배타적 bundle 저장을 지원하지 않으면 로컬 /content 경로를 사용하세요.


In [ ]:
UPDATE_DIAGNOSTIC_CHECKPOINT = ''  # @param {type:"string"}
UPDATE_BUNDLE_PATH = ''  # @param {type:"string"}
UPDATE_LR_REPORT = ''  # @param {type:"string"}

def run_update_diagnostic(command, environment):
    import signal
    process = subprocess.Popen(command, cwd=REPO_PATH, env=environment,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8',
        bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end='')
        return_code = process.wait()
    except KeyboardInterrupt:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
        raise
    finally:
        process.stdout.close()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

if UPDATE_DIAGNOSTIC_CHECKPOINT:
    import tempfile
    update_checkpoint = Path(UPDATE_DIAGNOSTIC_CHECKPOINT).expanduser().resolve()
    if not update_checkpoint.is_file():
        raise FileNotFoundError(f'업데이트 진단 체크포인트가 없습니다: {update_checkpoint}')
    diagnostics_dir = update_checkpoint.parent.parent / 'diagnostics'
    capture_stamp = str(time.time_ns())
    update_bundle = (Path(UPDATE_BUNDLE_PATH).expanduser().resolve() if UPDATE_BUNDLE_PATH else
                     Path('/content/stockbot_update_diagnostics') / ('update_comparison_' + capture_stamp + '.pt'))
    lr_report = (Path(UPDATE_LR_REPORT).expanduser().resolve() if UPDATE_LR_REPORT else
                 diagnostics_dir / ('update_lr_' + capture_stamp + '.json'))
    entry_report = lr_report.with_name(lr_report.stem + '_entry_credit.json')
    diagnostic_paths = (update_bundle, lr_report, entry_report)
    if len(set(diagnostic_paths)) != 3 or any(os.path.lexists(path) for path in diagnostic_paths):
        raise FileExistsError('진단 bundle과 보고서는 서로 다른 새 파일 경로를 사용하세요.')
    saved_update = torch.load(update_checkpoint, map_location='cpu', weights_only=True)
    capture_base = dict(CONFIG, load_policy=str(update_checkpoint),
        output_dir=str(diagnostics_dir), resume_lr=None, capture_update_bundle=None,
        total_timesteps=max(CONFIG['total_timesteps'], int(saved_update['total_timesteps']) + 1))
    capture_config = restore_run_config(capture_base, saved_update, 'resume')
    del saved_update
    capture_env = dict(os.environ, OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1',
                       SCALPING_TF32='0', PYTHONUNBUFFERED='1')
    with tempfile.TemporaryDirectory(prefix='stockbot_update_capture_') as temporary:
        capture_config_path = Path(temporary) / 'capture_config.json'
        capture_config_path.write_text(json.dumps(capture_config, indent=2), encoding='utf-8')
        capture_command = [sys.executable, '-u', '-m', 'ai_trader.grpo.train_xlstm',
            '--config', str(capture_config_path), '--load_policy', str(update_checkpoint), '--resume',
            '--capture_update_bundle', str(update_bundle), '--policy_update_checks']
        run_update_diagnostic(capture_command, capture_env)
    entry_command = [sys.executable, '-u', '-m', 'ai_trader.grpo.diagnose_entry_credit',
        '--bundle', str(update_bundle), '--output', str(entry_report)]
    run_update_diagnostic(entry_command, capture_env)
    print('BUY 손익·학습 신호 보고서:', entry_report)
    compare_command = [sys.executable, '-u', '-m', 'ai_trader.grpo.diagnose_update_lr',
        '--bundle', str(update_bundle), '--output', str(lr_report),
        '--learning-rates', '3e-5', '1e-5', '3e-6', '--device', 'auto']
    run_update_diagnostic(compare_command, capture_env)
    print('동일 rollout LR 비교 보고서:', lr_report)
    print(json.dumps(json.loads(lr_report.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
else:
    print('선택 검사: UPDATE_DIAGNOSTIC_CHECKPOINT를 지정하면 실행합니다.')


## 8. 학습 시작

기본 목표는 실제 수집 스텝 1,000,000개입니다. 학습 시간보다 비용 반영 검증 수익률을 우선하며, 검증은 64개 경로에서 평가합니다. 학습 예산 소진이 수익성 확보를 뜻하지는 않습니다.
회당 체크포인트를 Drive에 저장합니다. 중단된 학습은 `checkpoint_iter<N>.pt`를 명시하여 재개하세요.
`checkpoint_best.pt`는 validation에서 선택된 모델입니다. 평가 간격은 5회이며 마지막 회도 평가합니다.
재개 시 출발 모델과 호환되는 기존 best를 현재 validation에서 재평가하여 더 나은 모델을 보존합니다.
`resume`은 optimizer/누적 진행률을 이어가지만 난수 상태까지 동일한 비트 단위 재현을 보장하지 않습니다.

In [ ]:
import signal
from ai_trader.grpo.runtime_precision import precision_metadata

if GPU_PROBE is None or PROBED_CONFIG != json.dumps(CONFIG, sort_keys=True):
    raise RuntimeError('현재 CONFIG로 GPU 사전 점검을 먼저 실행하세요.')
if PROBED_LIKELIHOOD_SETTINGS != likelihood_probe_fingerprint(CONFIG, DIAGNOSTIC_CHECKPOINT, ENABLE_TF32):
    raise RuntimeError('현재 CONFIG/체크포인트/TF32 설정으로 7번 likelihood 검사를 다시 실행하세요.')
if MODE != 'resume' and OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError('이미 실행한 실험입니다. 새 RUN_NAME 또는 명시적인 resume으로 설정 셀부터 실행하세요.')
if estimated_host_gib(CONFIG) > 0.70 * available_ram_gib():
    raise MemoryError('학습 직전 여유 RAM이 부족합니다.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = OUTPUT_DIR / 'colab_config.json'
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding='utf-8')
run_info = dict(gpu=GPU_NAME, vram_gib=VRAM_GIB, available_ram_gib=available_ram_gib(),
                torch_version=str(torch.__version__), cuda_version=torch.version.cuda,
                code_revision=CODE_REVISION, source_sha256=SOURCE_SHA256, manifest_sha256=MANIFEST_SHA256,
                mode=MODE, enable_tf32=ENABLE_TF32, seed=SEED, gpu_probe=GPU_PROBE,
                observation_schema=OBSERVATION_SCHEMA,
                policy_likelihood_checks=POLICY_LIKELIHOOD_CHECKS, diagnostic_checkpoint=DIAGNOSTIC_CHECKPOINT,
                precision=precision_metadata())
(OUTPUT_DIR / 'colab_run.json').write_text(json.dumps(run_info, indent=2, ensure_ascii=False), encoding='utf-8')
child_env = dict(os.environ, OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1',
                 PYTHONUNBUFFERED='1', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
                 SCALPING_TF32='1' if ENABLE_TF32 else '0', SCALPING_SEED=str(SEED))
runner = GPU_SETUP + "\nfrom ai_trader.grpo.train_xlstm import main\nraise SystemExit(0 if main() else 1)\n"
command = [sys.executable, '-u', '-c', runner, '--config', str(CONFIG_PATH),
           '--policy_update_checks' if CONFIG['policy_update_checks'] else '--no-policy_update_checks',
           '--rollout_logprob_tolerance', str(CONFIG['rollout_logprob_tolerance']),
           '--kl_probe_samples', str(CONFIG['kl_probe_samples']),
           '--no_trade_max_validations', str(CONFIG['no_trade_max_validations'])]
if CONFIG.get('resume_lr') is not None:
    command.extend(['--resume_lr', str(CONFIG['resume_lr'])])
log_path = OUTPUT_DIR / f'colab_train_{time.strftime("%Y%m%d_%H%M%S")}.log'
with log_path.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(command, cwd=REPO_PATH, env=child_env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding='utf-8', bufsize=1,
                               start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
    except KeyboardInterrupt:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
        raise
    finally:
        process.stdout.close()
if return_code:
    raise subprocess.CalledProcessError(return_code, command)
print('학습 완료:', OUTPUT_DIR)

## 9. TensorBoard 및 비용 포함 평가 결과

`validation/mean_net_return`을 중심으로 KL, 거래 수, 보유 시간을 확인하세요.
학습마다 `checkpoints/diagnostics/entry_credit/iteration_*.json`에 BUY 손익·학습 신호를 저장합니다. TensorBoard의 `train/entry_credit/`에서 BUY advantage와 수익 거래의 음수 신호 비율을 확인할 수 있습니다. 거래 표본 수와 available 표시를 함께 보세요.
아래 수익률은 비용 포함 순자산 기준이며 미청산 포지션이 있으면 확정 실현 손익과 다릅니다.
`synthetic_execution_episodes`가 양수이면 실제 호가 재현이 아닌 합성 체결이 포함되었습니다.
test 결과로 하이퍼파라미터를 반복 선택하면 test도 검증 데이터가 되므로 새 홀드아웃이 필요합니다.
패턴 지표의 `success_rate`는 매수 체결 수량으로 가중한 **체결가 이상 도달률**입니다. 수수료 차감 이익 확률을 뜻하지 않습니다. `censored_quantity`는 데이터 부족으로 판정을 제외한 체결 수량입니다. 거래가 없으면 성공률은 미확정입니다.


In [ ]:
from IPython.display import display
import pandas as pd
from tensorboard import notebook as tb_notebook

tb_notebook.start('--logdir ' + str(OUTPUT_DIR / 'tensorboard_logs'))
report_path = OUTPUT_DIR / 'evaluation_report.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    if report.get('test_skip_reason'):
        print('Test skipped:', report['test_skip_reason'])
    fields = ['mean_net_return', 'mean_realized_net_pnl', 'mean_num_trades', 'max_drawdown',
              'avg_holding_time', 'incomplete_liquidation_episodes', 'max_open_quantity',
              'synthetic_execution_episodes', 'num_episodes']
    display(pd.DataFrame({part: {key: (report[part] or {}).get(key) for key in fields}
                          for part in ('validation', 'test')}))
    pattern_fields = ['success_quantity', 'failure_quantity', 'censored_quantity',
                      'labeled_buy_decisions', 'success_rate']
    print('매수 체결 후 1~5초 패턴 결과')
    display(pd.DataFrame({part: {key: ((report.get(part) or {}).get('entry_pattern') or {}).get(key)
                                for key in pattern_fields} for part in ('validation', 'test')}))
    print('체결 모델:', {part: (report[part] or {}).get('execution_models', 'not evaluated') for part in ('validation', 'test')})
else:
    print('평가 보고서는 학습 완료 후 생성됩니다. 중단된 경우 저장 체크포인트에서 재개하세요.')

## 10. 체결 가정 스트레스 검사 (선택)

`RUN_STRESS_TEST=True`로 실행합니다. **validation**에만 지연 250ms / 합성 스프레드 20bps /
슬리피지 5bps를 적용합니다. 실제 호가가 있으면 스프레드는 해당 호가를 따르므로 `spread_bps`는 합성 체결에만 적용됩니다.
비용 변화에 따른 수익률과 미청산 수량을 확인하세요. 최종 test 보고서는 학습 셀에서 생성한 것을 유지합니다.

In [ ]:
RUN_STRESS_TEST = False  # @param {type:"boolean"}
if RUN_STRESS_TEST:
    stress_path = OUTPUT_DIR / 'validation_execution_stress.json'
    stress_runner = GPU_SETUP + "\nfrom ai_trader.grpo.backtest import main\nraise SystemExit(main())\n"
    subprocess.run([sys.executable, '-u', '-c', stress_runner,
                    '--policy', str(OUTPUT_DIR / 'checkpoints/checkpoint_best.pt'),
                    '--extracted-dir', str(LOCAL_DATA_DIR), '--partition', 'validation',
                    '--episodes', str(CONFIG['evaluation_episodes']), '--seed', str(CONFIG['evaluation_seed']),
                    '--device', 'cuda', '--order-latency-ms', '250', '--spread-bps', '20',
                    '--slippage-bps', '5', '--output', str(stress_path)],
                   cwd=REPO_PATH, env=child_env, check=True)
    display(json.loads(stress_path.read_text(encoding='utf-8')))